<a href="https://colab.research.google.com/github/latikadhami05/ai-code-quality-risk-intelligence/blob/main/notebooks/V1_code_risk_estimator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/psf/requests.git

Cloning into 'requests'...
remote: Enumerating objects: 26870, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 26870 (delta 1), reused 0 (delta 0), pack-reused 26866 (from 3)
Receiving objects: 100% (26870/26870), 13.31 MiB | 16.15 MiB/s, done.
Resolving deltas: 100% (17618/17618), done.


In [2]:
!git clone https://github.com/pallets/flask.git
!git clone https://github.com/pallets/click.git

Cloning into 'flask'...
remote: Enumerating objects: 26353, done.
remote: Counting objects: 100% (94/94), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 26353 (delta 65), reused 44 (delta 44), pack-reused 26259 (from 2)
Receiving objects: 100% (26353/26353), 11.97 MiB | 10.65 MiB/s, done.
Resolving deltas: 100% (17613/17613), done.
Cloning into 'click'...
remote: Enumerating objects: 15734, done.
remote: Counting objects: 100% (719/719), done.
remote: Compressing objects: 100% (138/138), done.
remote: Total 15734 (delta 628), reused 581 (delta 581), pack-reused 15015 (from 2)
Receiving objects: 100% (15734/15734), 5.31 MiB | 15.10 MiB/s, done.
Resolving deltas: 100% (10781/10781), done.


In [5]:
!ls


click  flask  requests	sample_data


In [8]:
!ls requests

AUTHORS.rst  HISTORY.md  MANIFEST.in	 README.md	       src
docs	     LICENSE	 NOTICE		 requirements-dev.txt  tests
ext	     Makefile	 pyproject.toml  setup.py	       tox.ini


In [9]:
!ls flask

CHANGES.rst  examples	  pyproject.toml  src	 uv.lock
docs	     LICENSE.txt  README.md	  tests


In [10]:
import ast
import os
import pandas as pd

In [11]:
def find_python_files(folder):
    py_files = []
    for root, dirs, files in os.walk(folder):
        for file in files:
            if file.endswith(".py"):
                py_files.append(os.path.join(root, file))
    return py_files

requests_files = find_python_files("requests")
flask_files = find_python_files("flask")
click_files = find_python_files("click")

print(len(requests_files), "files in requests")
print(len(flask_files), "files in flask")
print(len(click_files), "files in click")

37 files in requests
83 files in flask
91 files in click


In [12]:
import warnings
warnings.filterwarnings("ignore")

def get_functions_from_file(filepath):
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            source = f.read()
        tree = ast.parse(source)
    except Exception:
        return []

    functions = []
    for node in ast.walk(tree):
        if isinstance(node, ast.FunctionDef):
            functions.append(node)
    return functions

sample_functions = get_functions_from_file(requests_files[0])
print("File:", requests_files[0])
print("Functions found:", len(sample_functions))
for fn in sample_functions:
    print("-", fn.name)

File: requests/setup.py
Functions found: 0


In [13]:
sample_functions = get_functions_from_file(requests_files[5])
print("File:", requests_files[5])
print("Functions found:", len(sample_functions))
for fn in sample_functions:
    print("-", fn.name)

File: requests/tests/test_structures.py
Functions found: 15
- setup
- test_list
- test_getitem
- test_delitem
- test_lower_items
- test_repr
- test_copy
- test_instance_equality
- setup
- test_repr
- test_getitem
- test_get
- test_hasattr
- test_getattr
- test_getattr_default


In [14]:
def get_function_features(node, source_lines):
    start = node.lineno
    end = max(child.lineno for child in ast.walk(node) if hasattr(child, "lineno"))
    loc = end - start + 1

    complexity = 1
    max_nesting = 0
    branches = 0
    variables = set()

    def walk_nesting(n, depth):
        nonlocal max_nesting
        max_nesting = max(max_nesting, depth)
        for child in ast.iter_child_nodes(n):
            if isinstance(child, (ast.If, ast.For, ast.While, ast.Try, ast.With)):
                walk_nesting(child, depth + 1)
            else:
                walk_nesting(child, depth)

    walk_nesting(node, 0)

    for child in ast.walk(node):
        if isinstance(child, (ast.If, ast.For, ast.While, ast.Try, ast.BoolOp, ast.ExceptHandler)):
            complexity += 1
            branches += 1
        if isinstance(child, ast.Name):
            variables.add(child.id)

    return {
        "function_name": node.name,
        "loc": loc,
        "complexity": complexity,
        "max_nesting": max_nesting,
        "branches": branches,
        "num_variables": len(variables),
        "num_args": len(node.args.args)
    }

# test it on the functions we found
for fn in sample_functions[:3]:
    print(get_function_features(fn, None))

{'function_name': 'setup', 'loc': 4, 'complexity': 1, 'max_nesting': 0, 'branches': 0, 'num_variables': 3, 'num_args': 1}
{'function_name': 'test_list', 'loc': 2, 'complexity': 1, 'max_nesting': 0, 'branches': 0, 'num_variables': 2, 'num_args': 1}
{'function_name': 'test_getitem', 'loc': 2, 'complexity': 1, 'max_nesting': 0, 'branches': 0, 'num_variables': 3, 'num_args': 2}


In [15]:
def build_dataset(file_list, repo_name):
    rows = []
    for filepath in file_list:
        functions = get_functions_from_file(filepath)
        for fn in functions:
            try:
                features = get_function_features(fn, None)
                features["file"] = filepath
                features["repo"] = repo_name
                rows.append(features)
            except Exception:
                continue
    return rows

requests_data = build_dataset(requests_files, "requests")
flask_data = build_dataset(flask_files, "flask")
click_data = build_dataset(click_files, "click")

all_data = requests_data + flask_data + click_data
df = pd.DataFrame(all_data)

print("Total functions extracted:", len(df))
df.head()

Total functions extracted: 4040


,function_name,loc,complexity,max_nesting,branches,num_variables,num_args,file,repo
0,test_request_url_handles_leading_path_separators,5,1,0,0,3,0,requests/tests/test_adapters.py,requests
1,echo_response_handler,10,1,0,0,5,1,requests/tests/test_lowlevel.py,requests
2,test_chunked_upload,13,1,1,0,11,0,requests/tests/test_lowlevel.py,requests
3,test_chunked_encoding_error,22,1,2,0,13,0,requests/tests/test_lowlevel.py,requests
4,test_chunked_upload_uses_only_specified_host_h...,16,1,1,0,14,0,requests/tests/test_lowlevel.py,requests


In [16]:
df.to_csv("v1_function_features.csv", index=False)
print("Saved", len(df), "rows to v1_function_features.csv")

Saved 4040 rows to v1_function_features.csv
